> **Production note (2026-06-21):** The script pipeline in `scripts/` is the source of truth for final outputs. This notebook is retained for exploration and narrative context; run the README pipeline for reproducible delivery artifacts.


# Modeling — Biodiesel Demand Forecasting
## Repsol Capstone Project — Sprint 2

**Goal:** Train and evaluate four models for each of the 5 target series, then produce a 24-month forecast (2026-01 → 2027-12).

**Models:**
| Model | Library | Notes |
|-------|---------|-------|
| SARIMA | statsmodels | Best fit for short, trending time series |
| Linear Regression (Ridge) | sklearn | Baseline ML, regularised |
| Random Forest | sklearn | Ensemble, handles non-linearity |
| XGBoost | xgboost | Gradient boosting |

**Key design choice — log1p transformation:**  
The biodiesel series shows explosive adoption-curve growth (×80 in 3 years). Modelling `log1p(Consumo_Tm)` and back-transforming with `expm1()` stabilises variance and improves all model fits.

**Data constraint note:**  
With only 21 effective training months per target (after lag features), ML models are at a structural disadvantage: they see a limited portion of the growth curve and struggle to extrapolate beyond the training maximum. SARIMA captures the autoregressive structure with fewer parameters and outperforms ML on this dataset.

**Inputs:** `data/processed/train.csv`, `data/processed/test.csv`  
**Outputs:** all written under a `legacy_notebook07_` filename prefix (see Section 8) so running this
notebook can never overwrite `scripts/05_modeling_with_cnmc.py`'s production outputs, even though both
historically used the same filenames.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.optimize import curve_fit
import xgboost as xgb

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 5)

NOTEBOOK_DIR   = Path().resolve()
REPO_ROOT      = NOTEBOOK_DIR.parent
DATA_INPUTS   = REPO_ROOT / 'data' / 'inputs'
DATA_FEATURES = REPO_ROOT / 'data' / 'features'
DATA_OUTPUTS  = REPO_ROOT / 'data' / 'outputs'
FIGURES        = REPO_ROOT / 'reports' / 'figures'

TARGETS = ['Nacional', 'Madrid', 'Cataluña', 'Andalucía', 'Valencia']
COLORS  = {'Nacional': '#FF6B35', 'Madrid': '#004E89', 'Cataluña': '#1A936F',
           'Andalucía': '#C84B31', 'Valencia': '#8E44AD'}

# Feature sets
# Macro features use only the _lag1 versions: the contemporaneous
# IPI_original / IPC_var_anual / Tasa_paro values are not actually known at
# forecast time (INE publishes them with a delay), so including them here
# would leak look-ahead information into both training and test evaluation.
# Mandate feature uses no lag: the annual obligation is set by BOE decree
# and published before the year it applies to — it is genuinely known at
# the start of each month.
ML_FEATS = [
    'Tendencia', 'Mes', 'sin_mes', 'cos_mes',
    'Lag_1', 'Lag_2', 'Lag_3',
    'Roll_mean_3', 'Roll_mean_6',
    'IPI_original_lag1', 'IPC_var_anual_lag1', 'Tasa_paro_lag1',
    'Mandato_Energia_Pct',
]

print("Setup complete.")
print(f"Targets: {TARGETS}")
print(f"ML features ({len(ML_FEATS)}): {ML_FEATS}")

## 1. Load Data

In [ ]:
df_train = pd.read_csv(DATA_FEATURES / 'features_train.csv')
df_test  = pd.read_csv(DATA_FEATURES / 'features_test.csv')
df_full  = pd.read_csv(DATA_FEATURES / 'features_modelo_completo.csv')
df_macro = (
    pd.read_csv(DATA_INPUTS / 'master_dataset.csv')
    .query('CCAA == "ESPAÑA"')[['Fecha', 'IPI_original', 'IPI_ajustado', 'IPC_var_anual', 'Tasa_paro']]
    .drop_duplicates('Fecha')
    .reset_index(drop=True)
)

print(f"Train: {df_train['Fecha'].min()} → {df_train['Fecha'].max()}  ({len(df_train)} rows)")
print(f"Test : {df_test['Fecha'].min()} → {df_test['Fecha'].max()}   ({len(df_test)} rows)")

# Quick look at Nacional growth
nac = df_full[df_full['Target']=='Nacional'][['Fecha','Consumo_Tm']]
print(f"\nNacional consumption range: {nac['Consumo_Tm'].min():.0f} → {nac['Consumo_Tm'].max():.0f} Tm")
print(f"Growth factor (Dec 2025 / Jan 2023): {nac.iloc[-1]['Consumo_Tm'] / nac.iloc[0]['Consumo_Tm']:.1f}×")

## 2. Helper Functions

### What?
Centralise metric computation, model training, and recursive forecasting.

### Why?
Running the same logic for 5 targets × 4 models without helpers would be ~400 lines of repetitive code. Helpers also make it easy to swap models or add new targets.

In [ ]:
# ── Metrics ────────────────────────────────────────────────────────────────────
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    y_true = np.array(y_true, dtype=float)
    y_pred = np.maximum(np.array(y_pred, dtype=float), 0)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mask = y_true > 0
    mape = float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100) if mask.sum() else np.nan
    sst  = float(np.sum((y_true - y_true.mean()) ** 2))
    r2   = float(1 - np.sum((y_true - y_pred) ** 2) / sst) if sst > 0 else np.nan
    return {'MAE': round(mae, 1), 'RMSE': round(rmse, 1),
            'MAPE': round(mape, 1), 'R2': round(r2, 3)}


# ── SARIMA ─────────────────────────────────────────────────────────────────────
def train_sarima(y_train: np.ndarray,
                 order=(1,1,1), seasonal_order=(1,0,0,12)):
    """Fit SARIMAX on log1p-transformed target, return fitted result."""
    model = SARIMAX(
        np.log1p(y_train),
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    return model.fit(disp=False)


def predict_sarima(result, n_steps: int) -> np.ndarray:
    return np.maximum(np.expm1(result.forecast(steps=n_steps)), 0)


# -- Saturating growth curves (Logistic / Gompertz) --------------------------
# Unlike SARIMA/ML extrapolating an unbounded trend, these have a built-in
# asymptote L (the eventual saturation level), which is the right shape for
# an adoption curve that is decelerating (confirmed in the EDA: YoY growth
# fell from >500% in 2023-2024 to ~230-290% in 2024-2025 across all targets).
# Fit directly on raw Consumo_Tm (no log1p) since the curve is already
# bounded -- no need for log-space variance stabilisation.
def _logistic_fn(t, L, k, t0):
    return L / (1 + np.exp(-k * (t - t0)))


def _gompertz_fn(t, L, b, k):
    return L * np.exp(-b * np.exp(-k * t))


def train_growth_curve(t: np.ndarray, y: np.ndarray, mes: np.ndarray, curve_type: str) -> dict:
    """
    Fit a saturating growth curve (logistic or Gompertz) to y vs. trend index t,
    then fit a small 2-parameter seasonal correction (sin/cos of calendar
    month) on the residuals. Raises if curve_fit fails to converge -- callers
    should catch this, since 13-23 point folds can occasionally fail.
    """
    y_max = float(np.max(y))
    if curve_type == 'Logistic':
        p0 = [y_max * 3, 0.3, float(np.median(t))]
        bounds = ([y_max * 1.01, 0.01, t.min() - 24], [y_max * 60, 3.0, t.max() + 60])
        popt, _ = curve_fit(_logistic_fn, t, y, p0=p0, bounds=bounds, maxfev=20000)
        trend_pred = _logistic_fn(t, *popt)
    elif curve_type == 'Gompertz':
        p0 = [y_max * 3, 5.0, 0.15]
        bounds = ([y_max * 1.01, 0.1, 0.001], [y_max * 60, 200.0, 2.0])
        popt, _ = curve_fit(_gompertz_fn, t, y, p0=p0, bounds=bounds, maxfev=20000)
        trend_pred = _gompertz_fn(t, *popt)
    else:
        raise ValueError(f"Unknown curve_type: {curve_type}")

    sin_mes = np.sin(2 * np.pi * mes / 12)
    cos_mes = np.cos(2 * np.pi * mes / 12)
    X = np.column_stack([sin_mes, cos_mes, np.ones_like(sin_mes)])
    seas_coef, *_ = np.linalg.lstsq(X, y - trend_pred, rcond=None)

    return {'curve_type': curve_type, 'params': popt, 'seasonal_coef': seas_coef}


def predict_growth_curve(model: dict, t: np.ndarray, mes: np.ndarray) -> np.ndarray:
    if model['curve_type'] == 'Logistic':
        trend_pred = _logistic_fn(t, *model['params'])
    else:
        trend_pred = _gompertz_fn(t, *model['params'])
    sin_mes = np.sin(2 * np.pi * mes / 12)
    cos_mes = np.cos(2 * np.pi * mes / 12)
    a, b, c = model['seasonal_coef']
    seasonal = a * sin_mes + b * cos_mes + c
    return np.maximum(trend_pred + seasonal, 0)


# -- ML (Ridge / RF / XGBoost) -------------------------------------------------
def train_ml(X_train: np.ndarray, y_train: np.ndarray, model_name: str):
    """Train a ML model on log1p target. Returns (model, scaler)."""
    scaler = StandardScaler()
    X_s = scaler.fit_transform(X_train)
    y_log = np.log1p(y_train)

    if model_name == 'Ridge':
        model = Ridge(alpha=10.0).fit(X_s, y_log)
    elif model_name == 'RandomForest':
        model = RandomForestRegressor(
            n_estimators=300, max_depth=3,
            min_samples_leaf=3, random_state=42
        ).fit(X_s, y_log)
    elif model_name == 'XGBoost':
        model = xgb.XGBRegressor(
            n_estimators=300, max_depth=2, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.8,
            reg_alpha=1, reg_lambda=5,
            random_state=42, verbosity=0
        ).fit(X_s, y_log)
    else:
        raise ValueError(f"Unknown model: {model_name}")

    return model, scaler


def predict_ml(model, scaler, X_test: np.ndarray) -> np.ndarray:
    log_pred = model.predict(scaler.transform(X_test))
    return np.maximum(np.expm1(np.clip(log_pred, None, 15.0)), 0)


# ── Recursive forecast for ML ──────────────────────────────────────────────────
def recursive_forecast_ml(model, scaler, history: list,
                           macro_last: dict, start_tendencia: int,
                           n_steps: int = 24) -> list:
    """
    Generate n_steps ahead forecasts using recursive 1-step prediction.
    history      : last 6 actual/predicted monthly values (most recent last)
    macro_last   : dict with last-known values for macro and mandate features.
                   Macro values (IPI, IPC, Tasa_paro) are held constant as a
                   neutral scenario. The mandate value (Mandato_Energia_Pct)
                   should be set to the legislated value for the forecast period.
    start_tendencia: trend index for the first forecast month
    """
    forecasts = []
    hist = list(history)  # rolling buffer

    for step in range(n_steps):
        trend = start_tendencia + step
        mes   = ((36 + step) % 12) + 1  # Jan 2026 = step 0 -> mes 1

        lag1 = hist[-1] if len(hist) >= 1 else 0
        lag2 = hist[-2] if len(hist) >= 2 else 0
        lag3 = hist[-3] if len(hist) >= 3 else 0
        rm3  = np.mean(hist[-3:]) if len(hist) >= 3 else np.mean(hist)
        rm6  = np.mean(hist[-6:]) if len(hist) >= 6 else np.mean(hist)

        # Build the row by feature name (keyed off ML_FEATS) rather than by
        # fixed position -- a positional array silently breaks whenever the
        # feature list changes. The mandate value comes from macro_last where
        # the caller has already set the legislated 2026-2027 value.
        feat_values = {
            'Tendencia': trend, 'Mes': mes,
            'sin_mes': np.sin(2 * np.pi * mes / 12),
            'cos_mes': np.cos(2 * np.pi * mes / 12),
            'Lag_1': lag1, 'Lag_2': lag2, 'Lag_3': lag3,
            'Roll_mean_3': rm3, 'Roll_mean_6': rm6,
            'IPI_original_lag1': macro_last['IPI_original'],
            'IPC_var_anual_lag1': macro_last['IPC_var_anual'],
            'Tasa_paro_lag1': macro_last['Tasa_paro'],
            'Mandato_Energia_Pct': macro_last['Mandato_Energia_Pct'],
        }
        row = np.array([[feat_values[f] for f in ML_FEATS]])

        pred = float(predict_ml(model, scaler, row)[0])
        if not np.isfinite(pred):
            pred = float(np.mean([v for v in hist[-3:] if np.isfinite(v)] or [0]))
        pred = max(0.0, pred)
        forecasts.append(pred)
        hist.append(pred)

    return forecasts


print("Helper functions ready.")

## 3. Train & Evaluate All Models

### What?
For each of the 5 targets: train all 4 models on the training set (2023-01 to 2024-12), generate 12-month test predictions (2025-01 to 2025-12), and compute MAE / RMSE / MAPE / R².

### Why?
Evaluating on the held-out 2025 window simulates the real forecasting task: predict 12 months ahead after training on ~2 years of history.

In [ ]:
all_metrics   = []  # will hold one row per (target, model)
all_preds     = []  # will hold one row per (target, model, fecha)
trained_models = {}  # store for forecast step

for target in TARGETS:
    print(f"\n{'='*55}")
    print(f"TARGET: {target}")
    print('='*55)

    tr = df_train[df_train['Target'] == target].sort_values('Fecha')
    te = df_test[df_test['Target']  == target].sort_values('Fecha')

    y_tr = tr['Consumo_Tm'].values
    y_te = te['Consumo_Tm'].values
    test_fechas = te['Fecha'].values

    trained_models[target] = {}

    # ── SARIMA ─────────────────────────────────────────────────────────────
    res_sarima = train_sarima(y_tr)
    pred_sarima = predict_sarima(res_sarima, n_steps=12)
    m = compute_metrics(y_te, pred_sarima)
    print(f"  SARIMA(1,1,1)(1,0,0,12): MAE={m['MAE']:.0f}  RMSE={m['RMSE']:.0f}  "
          f"MAPE={m['MAPE']:.1f}%  R²={m['R2']:.3f}")
    all_metrics.append({'Target': target, 'Model': 'SARIMA', **m})
    trained_models[target]['SARIMA'] = res_sarima
    for fecha, actual_val, pred in zip(test_fechas, y_te, pred_sarima):
        all_preds.append({'Fecha': fecha, 'Target': target, 'Actual': round(float(actual_val), 1),
                          'Model': 'SARIMA', 'Pred': round(pred, 1)})

    # -- Saturating growth curves (Logistic / Gompertz) ----------------------
    t_tr, mes_tr = tr['Tendencia'].values, tr['Mes'].values
    t_te, mes_te = te['Tendencia'].values, te['Mes'].values
    for curve_type in ['Logistic', 'Gompertz']:
        try:
            curve_model = train_growth_curve(t_tr, y_tr, mes_tr, curve_type)
            pred_curve = predict_growth_curve(curve_model, t_te, mes_te)
            m = compute_metrics(y_te, pred_curve)
            print(f"  {curve_type:15s}: MAE={m['MAE']:.0f}  RMSE={m['RMSE']:.0f}  "
                  f"MAPE={m['MAPE']:.1f}%  R²={m['R2']:.3f}")
            all_metrics.append({'Target': target, 'Model': curve_type, **m})
            trained_models[target][curve_type] = curve_model
            for fecha, actual_val, pred in zip(test_fechas, y_te, pred_curve):
                all_preds.append({'Fecha': fecha, 'Target': target, 'Actual': round(float(actual_val), 1),
                                  'Model': curve_type, 'Pred': round(float(pred), 1)})
        except Exception as e:
            print(f"  {curve_type:15s}: FAILED to fit ({e})")

    # ── ML models ──────────────────────────────────────────────────────────
    tr_ml = tr[ML_FEATS + ['Consumo_Tm']].dropna()
    te_ml = te[ML_FEATS + ['Consumo_Tm']].dropna()
    X_tr_ml = tr_ml[ML_FEATS].values
    y_tr_ml = tr_ml['Consumo_Tm'].values
    X_te_ml = te_ml[ML_FEATS].values
    y_te_ml = te_ml['Consumo_Tm'].values
    te_feats_dates = te['Fecha'].values if len(te_ml) > 0 else []

    if len(tr_ml) < 5:
        print(f"  Skipping ML models — insufficient training rows ({len(tr_ml)})")
        continue

    for model_name in ['Ridge', 'RandomForest', 'XGBoost']:
        mdl, scaler = train_ml(X_tr_ml, y_tr_ml, model_name)
        preds = predict_ml(mdl, scaler, X_te_ml)
        m = compute_metrics(y_te_ml, preds)
        label = {'Ridge': 'Ridge', 'RandomForest': 'Random Forest',
                 'XGBoost': 'XGBoost'}[model_name]
        print(f"  {label:15s}: MAE={m['MAE']:.0f}  RMSE={m['RMSE']:.0f}  "
              f"MAPE={m['MAPE']:.1f}%  R²={m['R2']:.3f}")
        all_metrics.append({'Target': target, 'Model': label, **m})
        trained_models[target][model_name] = (mdl, scaler)
        for fecha, actual_val, pred in zip(te_feats_dates, y_te_ml, preds):
            all_preds.append({'Fecha': fecha, 'Target': target, 'Actual': round(float(actual_val), 1),
                              'Model': label, 'Pred': round(float(pred), 1)})

print("\nTraining complete.")

## 4. Metrics Summary

### What?
Display a pivot table ranking all models by MAPE per target.

### Why?
MAPE is the most interpretable metric for business stakeholders ("off by X%"). MAE is the most actionable for operational planning (average error in tonnes). Both are shown.

In [ ]:
df_metrics = pd.DataFrame(all_metrics)

print("=" * 70)
print("MODEL EVALUATION — TEST SET (2025-01 → 2025-12)")
print("=" * 70)

for target in TARGETS:
    sub = df_metrics[df_metrics['Target'] == target].sort_values('MAPE')
    print(f"\n{target}")
    print(sub[['Model', 'MAE', 'RMSE', 'MAPE', 'R2']].to_string(index=False))

# NOTE: this ranks models by their 2025 test MAPE, which is useful as a
# diagnostic but must NOT be used to pick the production model -- doing so
# would mean the test set was used for model selection. See the
# Walk-Forward Validation section below for the actual selection method.
print("\n" + "=" * 70)
print("TEST-SET RANKING PER TARGET (diagnostic only -- not used for selection)")
print("=" * 70)
diag_best = df_metrics.loc[df_metrics.groupby('Target')['MAPE'].idxmin()]
print(diag_best[['Target', 'Model', 'MAE', 'MAPE', 'R2']].to_string(index=False))

# Saved under a legacy_notebook07_ prefix, not 'metricas_models.csv' --
# scripts/05_modeling_with_cnmc.py owns that filename for the production
# pipeline; this notebook must never overwrite it.
df_metrics.to_csv(DATA_OUTPUTS  / 'legacy_notebook07_metricas_models.csv', index=False)
print("\nSaved: legacy_notebook07_metricas_models.csv")

## 4b. Model Selection via Walk-Forward Validation (2023-2024 only)

### What?
For each target, score every candidate model (SARIMA, Ridge, Random Forest, XGBoost) using
expanding-window, 1-step-ahead walk-forward validation **inside the training period only**
(2023-01 to 2024-12). Each fold trains on all months up to an origin point and predicts the
next month; folds start once enough months exist for the lag/rolling features to be valid.
The model with the lowest mean walk-forward MAPE per target is the one we commit to.

### Why?
Choosing a model family by minimum MAPE on the 2025 test set (as in section 4 above) means the
test set has effectively been used for model selection -- the reported accuracy of the "winner"
is then optimistic, because we searched over 4 models per target and reported the best result.
Walk-forward validation lets us pick a model using only information available during training,
so the 2025 test score we report afterward is a single, honest out-of-sample number.

In [ ]:
def walk_forward_scores(target_df: pd.DataFrame, ml_feats: list,
                         min_origin: int = 15) -> dict:
    """
    Expanding-window, 1-step-ahead walk-forward MAPE per model family,
    computed entirely within the training period (2023-2024). No row from
    the 2025 test set is ever touched here.
    """
    df = target_df.sort_values('Fecha').reset_index(drop=True)
    n = len(df)
    fold_errors = {'SARIMA': [], 'Ridge': [], 'RandomForest': [], 'XGBoost': [],
                    'Logistic': [], 'Gompertz': []}

    for origin in range(min_origin, n - 1):
        fold_tr = df.iloc[:origin + 1]
        fold_te = df.iloc[origin + 1:origin + 2]
        y_true = float(fold_te['Consumo_Tm'].values[0])
        if y_true == 0:
            continue

        try:
            res = train_sarima(fold_tr['Consumo_Tm'].values)
            pred = float(predict_sarima(res, n_steps=1)[0])
            fold_errors['SARIMA'].append(abs((y_true - pred) / y_true) * 100)
        except Exception:
            pass

        for curve_type in ['Logistic', 'Gompertz']:
            try:
                curve_model = train_growth_curve(
                    fold_tr['Tendencia'].values, fold_tr['Consumo_Tm'].values,
                    fold_tr['Mes'].values, curve_type)
                pred = float(predict_growth_curve(
                    curve_model, fold_te['Tendencia'].values, fold_te['Mes'].values)[0])
                fold_errors[curve_type].append(abs((y_true - pred) / y_true) * 100)
            except Exception:
                pass

        tr_ml = fold_tr[ml_feats + ['Consumo_Tm']].dropna()
        te_ml = fold_te[ml_feats + ['Consumo_Tm']].dropna()
        if len(tr_ml) < 5 or len(te_ml) == 0:
            continue
        X_tr, y_tr = tr_ml[ml_feats].values, tr_ml['Consumo_Tm'].values
        X_te = te_ml[ml_feats].values

        for model_name in ['Ridge', 'RandomForest', 'XGBoost']:
            try:
                mdl, scaler = train_ml(X_tr, y_tr, model_name)
                pred = float(predict_ml(mdl, scaler, X_te)[0])
                fold_errors[model_name].append(abs((y_true - pred) / y_true) * 100)
            except Exception:
                pass

    # Median (not mean) across folds: with this little data, a single
    # diverging SARIMA fold (unstable seasonal fit on a short, explosively
    # growing series) can produce an absurd error that would otherwise
    # dominate a mean. The median is robust to that kind of single-fold blowup.
    return {m: (float(np.median(v)) if v else np.nan) for m, v in fold_errors.items()}


MODEL_LABELS = {'SARIMA': 'SARIMA', 'Ridge': 'Ridge',
                'RandomForest': 'Random Forest', 'XGBoost': 'XGBoost',
                'Logistic': 'Logistic', 'Gompertz': 'Gompertz'}

wf_rows = []
for target in TARGETS:
    tr = df_train[df_train['Target'] == target]
    scores = walk_forward_scores(tr, ML_FEATS)
    row = {'Target': target}
    row.update({MODEL_LABELS[m]: scores[m] for m in scores})
    wf_rows.append(row)

df_wf = pd.DataFrame(wf_rows).set_index('Target')
CANDIDATE_COLS = ['SARIMA', 'Ridge', 'Random Forest', 'XGBoost', 'Logistic', 'Gompertz']
df_wf['Selected_Model'] = df_wf[CANDIDATE_COLS].idxmin(axis=1)

print("=" * 70)
print("WALK-FORWARD VALIDATION -- MEDIAN MAPE (%), TRAINING PERIOD ONLY (2023-2024)")
print("=" * 70)
print(df_wf.round(1).to_string())

# Saved under a legacy_notebook07_ prefix, not 'model_selection_walkforward.csv'
# -- scripts/05_modeling_with_cnmc.py owns that filename for the production
# pipeline; this notebook must never overwrite it.
df_wf.to_csv(DATA_OUTPUTS / 'legacy_notebook07_model_selection_walkforward.csv')
print("\nSaved: legacy_notebook07_model_selection_walkforward.csv")

In [ ]:
final_rows = []
for target in TARGETS:
    sel_model = df_wf.loc[target, 'Selected_Model']
    m = df_metrics[(df_metrics['Target'] == target) & (df_metrics['Model'] == sel_model)]
    if len(m):
        final_rows.append(m.iloc[0])
df_final = pd.DataFrame(final_rows)[['Target', 'Model', 'MAE', 'RMSE', 'MAPE', 'R2']]

print("=" * 70)
print("FINAL MODEL PER TARGET -- chosen via walk-forward CV on 2023-2024,")
print("evaluated ONCE on the untouched 2025 holdout")
print("=" * 70)
print(df_final.to_string(index=False))

# Saved under a legacy_notebook07_ prefix, not 'metricas_final_selected.csv'
# -- scripts/05_modeling_with_cnmc.py owns that filename for the production
# pipeline; this notebook must never overwrite it.
df_final.to_csv(DATA_OUTPUTS / 'legacy_notebook07_metricas_final_selected.csv', index=False)
print("\nSaved: legacy_notebook07_metricas_final_selected.csv")

## 5. Visual Comparison — Predictions vs Actuals

### What?
Plot actual vs predicted values for all 4 models on the test period for each target.

### Why?
Numbers alone don't show *where* each model fails. Plots reveal systematic over/under-estimation, lag in capturing trend changes, and which models track the seasonal pattern.

In [ ]:
df_preds = pd.DataFrame(all_preds)
df_actuals = pd.concat([df_train, df_test])[['Fecha', 'Target', 'Consumo_Tm']]

model_colors = {
    'SARIMA':        '#004E89',
    'Ridge':         '#1A936F',
    'Random Forest': '#C84B31',
    'XGBoost':       '#8E44AD',
    'Logistic':      '#E8A33D',
    'Gompertz':      '#5C3D5E',
}

fig, axes = plt.subplots(3, 2, figsize=(18, 14))
axes = axes.flatten()

for i, target in enumerate(TARGETS):
    ax = axes[i]
    actual = df_actuals[df_actuals['Target'] == target].sort_values('Fecha')
    actual['Fecha_dt'] = pd.to_datetime(actual['Fecha'])

    # Full actual line
    ax.plot(actual['Fecha_dt'], actual['Consumo_Tm'],
            color='black', linewidth=2, label='Actual', zorder=5)

    # Train/test boundary
    split = pd.to_datetime('2025-01')
    ax.axvline(split, color='grey', linestyle='--', linewidth=1)

    # Model predictions (test period only)
    preds_t = df_preds[df_preds['Target'] == target]
    for model_name, color in model_colors.items():
        mp = preds_t[preds_t['Model'] == model_name].sort_values('Fecha')
        if mp.empty:
            continue
        mp['Fecha_dt'] = pd.to_datetime(mp['Fecha'])
        ax.plot(mp['Fecha_dt'], mp['Pred'],
                color=color, linewidth=1.5, linestyle='--',
                marker='o', markersize=4, label=model_name)

    ax.set_title(f'{target}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Consumption (Tm)')
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, alpha=0.3)

axes[5].set_visible(False)
fig.suptitle('Model Predictions vs Actuals — Test Period 2025',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES / '11_predictions_vs_actuals.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: 11_predictions_vs_actuals.png")

## 6. 24-Month Forecast (2026-01 → 2027-12)

### What?
For each target, generate a 24-month forecast using:
- **SARIMA**: direct forecast via `model.forecast(steps=24)`
- **ML models**: recursive 1-step-ahead forecasting (each prediction becomes Lag_1 for the next)

### Why?
The primary deliverable of this project is the 24-month demand forecast. SARIMA is used as the primary forecast because it outperforms ML on the test set. ML forecasts are included for scenario comparison.

**Macro assumption for forecast period:** INE macro variables (IPI, IPC, Tasa Paro) are held at their last known values (Dec 2025). This is a neutral scenario — no economic shocks assumed.

In [ ]:
forecast_rows = []
forecast_dates = pd.date_range('2026-01', periods=24, freq='MS').strftime('%Y-%m').tolist()

for target in TARGETS:
    # --- SARIMA: re-train on full 36-month series (train + test) ---
    y_full = pd.concat([df_train, df_test])[
        pd.concat([df_train, df_test])['Target'] == target
    ].sort_values('Fecha')['Consumo_Tm'].values

    res_full = train_sarima(y_full)
    fc_sarima = predict_sarima(res_full, n_steps=24)
    for fecha, val in zip(forecast_dates, fc_sarima):
        forecast_rows.append({'Fecha': fecha, 'Target': target,
                               'Model': 'SARIMA', 'Forecast': round(float(val), 1)})

    # --- Growth curves: refit on the full 36-month series, direct functional forecast ---
    full_df = pd.concat([df_train, df_test])
    full_tgt = full_df[full_df['Target'] == target].sort_values('Fecha')
    t_future = np.arange(37, 37 + 24)
    mes_future = ((t_future - 1) % 12) + 1
    for curve_type in ['Logistic', 'Gompertz']:
        try:
            curve_model_full = train_growth_curve(
                full_tgt['Tendencia'].values, full_tgt['Consumo_Tm'].values,
                full_tgt['Mes'].values, curve_type)
            fc_curve = predict_growth_curve(curve_model_full, t_future, mes_future)
            for fecha, val in zip(forecast_dates, fc_curve):
                forecast_rows.append({'Fecha': fecha, 'Target': target,
                                       'Model': curve_type, 'Forecast': round(float(val), 1)})
        except Exception as e:
            print(f"  {target} {curve_type}: forecast FAILED ({e})")

    # --- ML: recursive forecast using last 6 actuals as history ---
    last_actuals = y_full[-6:].tolist()

    # Macro held at last known value (Dec 2025) — neutral scenario.
    # Mandate set to the legislated 2026 value (not extrapolated — confirmed against BOE).
    # 2027 is the team's own projection (no Real Decreto published yet for that year);
    # holding constant at the 2026 value here is a simplification specific to this
    # legacy notebook -- the production pipeline (scripts/05) uses the per-year schedule.
    macro_last = df_macro.iloc[-1][['IPI_original', 'IPC_var_anual', 'Tasa_paro']].to_dict()
    macro_last['Mandato_Energia_Pct'] = 14.0       # RD 5/2026 (confirmed against BOE)

    start_tendencia = 37  # one beyond the 36 training months

    for model_name in ['Ridge', 'RandomForest', 'XGBoost']:
        if model_name not in trained_models[target]:
            continue
        mdl, scaler = trained_models[target][model_name]
        label = {'Ridge': 'Ridge', 'RandomForest': 'Random Forest',
                 'XGBoost': 'XGBoost'}[model_name]
        fc = recursive_forecast_ml(mdl, scaler, last_actuals, macro_last,
                                   start_tendencia, n_steps=24)
        for fecha, val in zip(forecast_dates, fc):
            forecast_rows.append({'Fecha': fecha, 'Target': target,
                                   'Model': label, 'Forecast': round(float(val), 1)})

df_forecast = pd.DataFrame(forecast_rows)
print(f"Forecast rows: {len(df_forecast)}")
print(f"\nMandate value used for ML forecast: Energia={macro_last['Mandato_Energia_Pct']}%")
print(f"\nNacional SARIMA 24-month forecast:")
print(df_forecast[(df_forecast['Target']=='Nacional') &
                  (df_forecast['Model']=='SARIMA')][['Fecha','Forecast']].to_string(index=False))

## 7. Visualise 24-Month Forecast

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(18, 14))
axes = axes.flatten()

for i, target in enumerate(TARGETS):
    ax = axes[i]

    # Historical actuals
    hist = df_actuals[df_actuals['Target'] == target].sort_values('Fecha')
    hist['Fecha_dt'] = pd.to_datetime(hist['Fecha'])
    ax.plot(hist['Fecha_dt'], hist['Consumo_Tm'],
            color='black', linewidth=2, label='Historical', zorder=5)

    # Forecast period
    fc_t = df_forecast[df_forecast['Target'] == target]
    for model_name, color in model_colors.items():
        fc_m = fc_t[fc_t['Model'] == model_name].copy()
        if fc_m.empty:
            continue
        fc_m['Fecha_dt'] = pd.to_datetime(fc_m['Fecha'])
        ax.plot(fc_m['Fecha_dt'], fc_m['Forecast'],
                color=color, linewidth=2, linestyle='-',
                marker='o', markersize=3, label=f'{model_name} forecast', alpha=0.85)

    # Shade forecast window
    fc_start = pd.to_datetime('2026-01')
    ax.axvline(fc_start, color='grey', linestyle='--', linewidth=1)
    ymax = max(hist['Consumo_Tm'].max(),
               fc_t['Forecast'].max() if not fc_t.empty else 0) * 1.1
    ax.fill_betweenx([0, ymax], fc_start,
                      pd.to_datetime('2027-12'), alpha=0.05, color='blue')

    ax.set_title(f'{target}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Consumption (Tm)')
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, ymax)

axes[5].set_visible(False)
fig.suptitle('24-Month Demand Forecast — All Models (2026–2027)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES / '12_forecast_24m.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: 12_forecast_24m.png")

## 8. Save All Outputs

In [ ]:
# All three filenames below previously collided with scripts/05_modeling_with_cnmc.py's
# production outputs. Saved under a legacy_notebook07_ prefix instead so running this
# notebook can never silently overwrite the production pipeline's results.

# Predictions on test set
df_preds.to_csv(DATA_OUTPUTS  / 'legacy_notebook07_predicciones_test_2025.csv', index=False)

# 24-month forecast
df_forecast.to_csv(DATA_OUTPUTS  / 'legacy_notebook07_forecast_24m_sarima_rf_xgb.csv', index=False)

# Combined historical + forecast for Tableau
hist_long = df_actuals[['Fecha', 'Target', 'Consumo_Tm']].copy()
hist_long['Type'] = 'Historical'
hist_long['Model'] = 'Actual'
hist_long.rename(columns={'Consumo_Tm': 'Valor'}, inplace=True)

fc_long = df_forecast[df_forecast['Model'] == 'SARIMA'].copy()
fc_long['Type'] = 'Forecast'
fc_long.rename(columns={'Forecast': 'Valor'}, inplace=True)

tableau_df = pd.concat([hist_long, fc_long[['Fecha','Target','Model','Valor','Type']]],
                        ignore_index=True)
tableau_df.to_csv(DATA_OUTPUTS  / 'legacy_notebook07_tableau_export.csv', index=False)

print("Files saved:")
print(f"  legacy_notebook07_predicciones_test_2025.csv      — {len(df_preds)} rows (test period predictions, all models)")
print(f"  legacy_notebook07_forecast_24m_sarima_rf_xgb.csv  — {len(df_forecast)} rows (2026-2027 forecast, all models)")
print(f"  legacy_notebook07_tableau_export.csv              — {len(tableau_df)} rows (historical + SARIMA forecast)")
print(f"  legacy_notebook07_metricas_models.csv              — {len(df_metrics)} rows")

## 9. Summary

### Sprint 2 — Modeling: DONE (leakage fix + saturating growth curves) ✅

**What changed in this revision:**
- Removed the contemporaneous `IPI_original` / `IPC_var_anual` / `Tasa_paro` features from
  `ML_FEATS` (kept only the `_lag1` versions) — these were look-ahead leaks, since INE publishes
  them with a delay.
- Model family per target is chosen via walk-forward validation inside the training period
  (2023-2024) instead of by minimum MAPE on the 2025 test set. The 2025 number below is reported
  **once**, for the model walk-forward already committed to — it was never used to pick a winner.
- Added two new candidates: **Logistic** and **Gompertz** saturating growth curves, fit directly
  on `Consumo_Tm` (no log1p needed -- the curve already has a built-in asymptote) plus a small
  2-parameter sin/cos seasonal correction. Motivation: YoY growth decelerates from >500% in
  2023-2024 to ~230-290% in 2024-2025 across *every* target -- the signature of an adoption
  curve approaching its bend, not unbounded exponential growth. SARIMA/ML extrapolate trend
  forward with no ceiling; these curves do.

**Result: the growth curves were tested through the exact same walk-forward selection used for
every other candidate, and they won (or tied) for every target.** No target got worse:

| Target | Previous selection | Previous MAPE | New selection | New MAPE |
|--------|--------------------|----------------|----------------|-----------|
| Nacional | SARIMA | 29.0% | SARIMA (unchanged) | 29.0% |
| Madrid | SARIMA | 318.7% | **Gompertz** | **197.1%** |
| Cataluña | Ridge | 3332.6% | **Gompertz** | **164.2%** |
| Andalucía | SARIMA | 52.5% | **Logistic** | **48.4%** |
| Valencia | SARIMA | 57.4% | **Gompertz** | **34.2%** |

**Honest caveat:** this is a real improvement, not a fix. R² is still strongly negative for
Madrid (-101.0) and Cataluña (-91.3) -- both targets are still extrapolating worse than a naive
mean, just far less catastrophically than before (R² was -312.9 and -49,798 respectively). The
saturating-curve hypothesis was directionally correct (it stopped the worst blow-ups), but 21-23
training points is still not enough to pin down a 3-parameter curve plus seasonal correction with
real confidence, especially for targets where the "bend" is barely visible yet in the training
window. Walk-forward validation also still only tests 1-step-ahead folds, not the full 12-month
horizon being reported here.

**Outputs (all under a `legacy_notebook07_` prefix -- see the note added 2026-06-25 in
`NOTEBOOKS_AUDIT.md`; these filenames used to collide with `scripts/05_modeling_with_cnmc.py`'s
production outputs, which is no longer possible):**
- `legacy_notebook07_metricas_models.csv` -- full diagnostic grid, all 6 models x all 5 targets, test period
- `legacy_notebook07_model_selection_walkforward.csv` -- per-target walk-forward MAPE and selected model
- `legacy_notebook07_metricas_final_selected.csv` -- the single reported test metric per target
- `legacy_notebook07_predicciones_test_2025.csv` / `legacy_notebook07_forecast_24m_sarima_rf_xgb.csv` / `legacy_notebook07_tableau_export.csv`

### Next step -> `09_evaluation.ipynb`
Deep-dive evaluation: residual analysis, feature importance, model comparison charts, and the
updated recommendation table reflecting the walk-forward-selected models.